# 第 1 章 · PyTorch 做了什么：一行 `y = x @ w` 的旅程

本章回答：numpy 也能做矩阵乘法，PyTorch 多做了什么？你写下一行 `y = x @ w`，它在 PyTorch 里面经过了哪几站？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/01-op-journey.md`（讲解、动画、练习都在讲义里）。对应原文：[00-overview.md](../00-overview.md) §1–§2.2

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 本机 GPU 后端 | — | `torch.backends.mps.is_available()` | Mac 上是 mps |
| 第二步 要梯度 | 账本记一笔 | `journey(True)[1]` | `MmBackward0` |
| 第二步 不要梯度 | 记账员放行 | `journey(False)[1]` | `None` |
| 第二步 inference_mode | 整段不记账 | `journey_inference()[1]` | `None` |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `journey`**（来自 `ch01_op_journey.py`）：算一次 y = x @ w，返回 y 和它身上的 grad_fn（账本上的那一笔）。

In [ ]:
def journey(requires_grad: bool, device: str = "cpu"):
    """算一次 y = x @ w，返回 y 和它身上的 grad_fn（账本上的那一笔）。"""
    torch.manual_seed(0)
    x = torch.randn(2, 3, device=device)
    w = torch.randn(3, 4, device=device, requires_grad=requires_grad)
    y = x @ w
    return y, y.grad_fn

**函数 `journey_inference`**（来自 `ch01_op_journey.py`）：在 inference_mode 里算同一行：整段都不记账。

In [ ]:
def journey_inference(device: str = "cpu"):
    """在 inference_mode 里算同一行：整段都不记账。"""
    torch.manual_seed(0)
    x = torch.randn(2, 3, device=device)
    w = torch.randn(3, 4, device=device, requires_grad=True)
    with torch.inference_mode():
        y = x @ w
    return y, y.grad_fn

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 numpy 能算，PyTorch 多了 GPU 和自动求导")
dev = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("本机可用的 GPU 后端：", dev)

In [ ]:
print("\n第二步 四站：前端 → 分派器 → autograd 记一笔 → kernel")
y, fn = journey(requires_grad=True)
print("w 要梯度：y.grad_fn =", type(fn).__name__)
y, fn = journey(requires_grad=False)
print("w 不要梯度：y.grad_fn =", fn)
y, fn = journey_inference()
print("inference_mode 里：y.grad_fn =", fn)

## 3. 代码里才看得到的

1. **`grad_fn` 的类名后面带 0**：`MmBackward0` 里的 0 是版本号，不是第 0 个节点。
2. **`device` 一个参数就换厨房**：同一个 `journey` 把 `device` 换成 `"mps"` 就走 GPU，函数体一个字不用改。

## 4. 自己改一改

- 把 `journey` 里的 `x` 也设成 `requires_grad=True`，看 `y.grad_fn` 变不变。
- 在 Mac 上调 `journey(True, "mps")`，看结果张量的 `device`。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> numpy 只会炒，PyTorch 开餐厅：多个厨房（GPU），一个记账员（autograd）；
> 一行代码四站走：服务员接单，前台看标签，记账员记一笔，厨房来下锅。